# Sobre o Trabalho
O Presente Trabalho buscou dados abertos de exportação do Governo Federal para realizar analise das exportações de armamento por parte do Brasil, foram contemplados o ano 2000 até o ano 2025 para a presente analise.

## Observações
- Muitas visualizações foram feitas em Plotly, ao exportar o arquivo jupyter seus outputs não aparecem, assimforam salvos em png, colocados juntamente com esse notebook no github e onde deveriam ser visualizados faz-se referencia ao nomeda figura em markdown.

- As evidencias do catalago de dados de cada tabela tambem esta como pngs, para cada tabela tem um png do overview e outro de informações das colunas.

- Este é o unico notebook, os demais arquivos são apenas os pngs supracitados.


# Perguntas a Responder
- O valor global das exportaçoes de armamento tem mudado❔ Se sim, como❔Qual a tendencia atual❔
- Quais os três paises que historicamente mais compraram do Brasil❔ Percentualmente quanto isso representa de nossas vendas❔ Juntos mantem um padrão de compra percentual ao longo dos anos❔
- Quais os três produtos mais vendidos pelo Brasil❔ Percentualmente quanto isso representa de nossas vendas❔ Juntos mantem um padrão de compra percentual ao longo dos anos❔
- Historicamente quais as principais vias de exportação❔
- Existe algum mês do ano em que as vendas tendem a ocorrer mais❔

# Criação do Catalago e Bancos de Dados
Aqui é Criado o Catalago de nome MVP e os Bancos de Dados Bronze, Prata e Ouro

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS MVP;
USE CATALOG MVP;
CREATE SCHEMA IF NOT EXISTS Bronze;
CREATE SCHEMA IF NOT EXISTS Prata;
CREATE SCHEMA IF NOT EXISTS Ouro;




In [0]:
import plotly.express as px
import numpy as np
from pyspark.sql.functions import col, concat, lit, to_date, lpad
from pyspark.sql import functions as F
import pandas as pd
import requests
import urllib3
import io
from datetime import datetime
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Ingestão de Dados

## Webscraping para criação da tabela Fato nivel Bronze, tal tabela corresponde ao registro de exportações do ano de 2000 até 2025, todos os dados foram interpretados como texto

In [0]:


from pyspark.sql.utils import AnalysisException

table_name = "MVP.Bronze.Exportacoes"
first_write = True

for i in range(2000, 2026):
    url = f'https://balanca.mdic.gov.br/balanca/bd/comexstat-bd/ncm/EXP_{i}.csv'
    try:
        r = requests.get(url, verify=False, timeout=120)
        r.raise_for_status()
    except Exception as e:
        print(f"Falhou ano {i}: {e}")
        continue

    df = pd.read_csv(io.StringIO(r.text), sep=";", encoding="latin1", dtype=str)
    df_spark = spark.createDataFrame(df)

    mode = "overwrite" if first_write else "append"
    (df_spark.write
        .format("delta")
        .mode(mode)
        .option("mergeSchema", "true")
        .saveAsTable(table_name))

    first_write = False
    print(f"Ano {i} gravado ({len(df)} linhas)")

Ano 2000 gravado (608920 linhas)
Ano 2001 gravado (642737 linhas)
Ano 2002 gravado (652868 linhas)
Ano 2003 gravado (756433 linhas)
Ano 2004 gravado (872417 linhas)
Ano 2005 gravado (1082314 linhas)
Ano 2006 gravado (1060518 linhas)
Ano 2007 gravado (1107634 linhas)
Ano 2008 gravado (1085591 linhas)
Ano 2009 gravado (1006104 linhas)
Ano 2010 gravado (1044917 linhas)
Ano 2011 gravado (1071261 linhas)
Ano 2012 gravado (1063549 linhas)
Ano 2013 gravado (1074709 linhas)
Ano 2014 gravado (1057794 linhas)
Ano 2015 gravado (1048388 linhas)
Ano 2016 gravado (1092692 linhas)
Ano 2017 gravado (1096623 linhas)
Ano 2018 gravado (1128981 linhas)
Ano 2019 gravado (1393174 linhas)
Ano 2020 gravado (1344556 linhas)
Ano 2021 gravado (1480295 linhas)
Ano 2022 gravado (1495868 linhas)
Ano 2023 gravado (1563659 linhas)
Ano 2024 gravado (1603796 linhas)
Ano 2025 gravado (1709746 linhas)


In [0]:
%sql
SELECT * FROM MVP.Prata.Exportacoes LIMIT 10


CO_ANO,CO_MES,CO_NCM,CO_UNID,CO_PAIS,SG_UF_NCM,CO_VIA,CO_URF,QT_ESTAT,KG_LIQUIDO,VL_FOB,data
2007,04,02071400,10,741,SC,01,0920600,633233.0,2683822.0,3699884.0,2007-04-01
2007,04,64039990,13,687,RS,01,1010500,444.0,218.0,4943.0,2007-04-01
2007,04,56011000,10,169,SP,01,0817800,28.0,4753.0,43626.0,2007-04-01
2007,04,87089990,10,249,MG,01,0817800,6678.0,123863.0,124326.0,2007-04-01
2007,04,85364100,11,493,SP,01,0817800,92688.0,4605.0,145403.0,2007-04-01
2007,04,16023100,10,386,PR,01,0910700,24000.0,24000.0,86794.0,2007-04-01
2007,04,85334099,11,158,SP,07,1010900,24.0,1.0,346.0,2007-04-01
2007,04,90328929,11,196,SP,04,0817700,4.0,1.0,569.0,2007-04-01
2007,04,42021100,11,063,MN,07,1010900,32.0,16.0,644.0,2007-04-01
2007,04,84133020,11,391,SP,04,0817700,1.0,1.0,38.0,2007-04-01


## Tabelas Dimeções Nivel Bronze
Embora não seja tão comum usar dimeções no nivel bronze, uma vez que o governo as entrega e são nescessarias para a analise da Fato tambem foram adquiridas via webscrapping com os dados interpretados como texto

In [0]:
r=requests.get('https://balanca.mdic.gov.br/balanca/bd/tabelas/VIA.csv', verify=False)
via=pd.read_csv(io.StringIO(r.text), sep=";",encoding='latin1',dtype=str)
via_spark = spark.createDataFrame(via)
via_spark.write.format("delta").mode("overwrite").saveAsTable("MVP.Bronze.DimVia")

r=requests.get('https://balanca.mdic.gov.br/balanca/bd/tabelas/URF.csv', verify=False)
urf=pd.read_csv(io.StringIO(r.text), sep=";",encoding='latin1',dtype=str)
urf_spark = spark.createDataFrame(urf)
urf_spark.write.format("delta").mode("overwrite").saveAsTable("MVP.Bronze.DimURF")

r=requests.get('https://balanca.mdic.gov.br/balanca/bd/tabelas/PAIS.csv', verify=False)
pais=pd.read_csv(io.StringIO(r.text), sep=";",encoding='latin1',dtype=str)
pais_spark = spark.createDataFrame(pais)
pais_spark.write.format("delta").mode("overwrite").saveAsTable("MVP.Bronze.DimPais")

r=requests.get('https://balanca.mdic.gov.br/balanca/bd/tabelas/NCM_UNIDADE.csv', verify=False)
unidade=pd.read_csv(io.StringIO(r.text), sep=";",encoding='latin1',dtype=str)
unidade_spark = spark.createDataFrame(unidade)
unidade_spark.write.format("delta").mode("overwrite").saveAsTable("MVP.Bronze.DimUnidade")

r=requests.get('https://balanca.mdic.gov.br/balanca/bd/tabelas/NCM.csv', verify=False)
ncm=pd.read_csv(io.StringIO(r.text), sep=";",encoding='latin1',dtype=str)
ncm_spark = spark.createDataFrame(ncm)
ncm_spark.write.format("delta").mode("overwrite").saveAsTable("MVP.Bronze.DimNCM")



# Analise de Problemas nas Tabelas
- Inicialmente foram verificados a existencia de valores nulos nas, apenas a tabela dimncm possuia valores nulos, mas ao verificar as colunas que os mesmos ocorriam elas foram consideradas não criticas para a analise em questão e tais valores puderam ser mantidos sem problemas.
- Em seguida a consistencia das colunas utilizadas como FK na tabela Fato e PK nas tabelas dimenções foram analisadas, todas se mostram consistentes.
- Por ultimo foi verificado duplicidade de PK nas tabelas dimenções. Não foram observadas.

In [0]:
tabelas = [t.name for t in spark.catalog.listTables("MVP.Bronze")]
for tabela in tabelas:
    df = spark.table(f"MVP.Ouro.{tabela}")
    print(f"--- {tabela} ---")
    print(f'Total de Valores Brancos= {df.toPandas().isnull().sum().sum()}')

--- dimncm ---
Total de Valores Brancos= 1248
--- dimpais ---
Total de Valores Brancos= 0
--- dimunidade ---
Total de Valores Brancos= 0
--- dimurf ---
Total de Valores Brancos= 0
--- dimvia ---
Total de Valores Brancos= 0
--- exportacoes ---
Total de Valores Brancos= 0


In [0]:
df = spark.table(f"MVP.Ouro.dimncm")
df.toPandas().isnull().sum()


CO_NCM              0
CO_UNID             0
CO_SH6              0
CO_PPE            306
CO_PPI            306
CO_FAT_AGREG       22
CO_CUCI_ITEM        0
CO_CGCE_N3          0
CO_SIIT           306
CO_ISIC_CLASSE      0
CO_EXP_SUBSET     308
NO_NCM_POR          0
NO_NCM_ESP          0
NO_NCM_ING          0
dtype: int64

In [0]:
print('----Quantidade de Problemas de Integridade Encontrados-----')
df=spark.table("MVP.Bronze.Exportacoes").toPandas()
print(f'Exportacoes NCM= {len(df[df['CO_NCM'].str.len()!=8])}')
print(f'Exportacoes UNID= {len(df[df['CO_UNID'].str.len()!=2])}')
print(f'Exportacoes PAIS= {len(df[df['CO_PAIS'].str.len()!=3])}')
print(f'Exportacoes VIA= {len(df[df['CO_VIA'].str.len()!=2])}')
print(f'Exportacoes URF= {len(df[df['CO_URF'].str.len()!=7])}')

df=spark.table("MVP.Bronze.dimncm").toPandas()
print(f'dimncm NCM= {len(df[df['CO_NCM'].str.len()!=8])}')

df=spark.table("MVP.Bronze.dimunidade").toPandas()
print(f'dimunidade UNID= {len(df[df['CO_UNID'].str.len()!=2])}')

df=spark.table("MVP.Bronze.dimpais").toPandas()
print(f'dimpais PAIS= {len(df[df['CO_PAIS'].str.len()!=3])}')

df=spark.table("MVP.Bronze.dimvia").toPandas()
print(f'dimvia VIA= {len(df[df['CO_VIA'].str.len()!=2])}')

df=spark.table("MVP.Bronze.dimurf").toPandas()
print(f'dimurf URF= {len(df[df['CO_URF'].str.len()!=7])}')

----Quantidade de Problemas de Integridade Encontrados-----
Exportacoes NCM= 0
Exportacoes UNID= 0
Exportacoes PAIS= 0
Exportacoes VIA= 0
Exportacoes URF= 0
dimncm NCM= 0
dimunidade UNID= 0
dimpais PAIS= 0
dimvia VIA= 0
dimurf URF= 0


In [0]:
print('----Verificação de Unicidade----')
df=spark.table("MVP.Bronze.dimncm").toPandas()
print(f'dimncm NCM= {df['CO_NCM'].is_unique}')

df=spark.table("MVP.Bronze.dimunidade").toPandas()
print(f'dimunidade UNID= {df['CO_UNID'].is_unique}')

df=spark.table("MVP.Bronze.dimpais").toPandas()
print(f'dimpais PAIS= {df['CO_PAIS'].is_unique}')

df=spark.table("MVP.Bronze.dimvia").toPandas()
print(f'dimvia VIA= {df['CO_VIA'].is_unique}')

df=spark.table("MVP.Bronze.dimurf").toPandas()
print(f'dimurf URF= {df['CO_URF'].is_unique}')

----Verificação de Unicidade----
dimncm NCM= True
dimunidade UNID= True
dimpais PAIS= True
dimvia VIA= True
dimurf URF= True


# Criação das Tabelas Prata
Uma vez que as tabelas foram consideradas integras foi verificada a nescessidade de alteração de tipagem ou novas colunas antes de sua passagem para o banco de dados Prata. A unica tabela que foi verificada alguma nescessidade de alteração foi a tabela Fato de exportação. Nela foi criada uma coluna de data do tipo date, as colunas QT_ESTAT, KG_LIQUIDO e VL_FOB foram convertidas para float e as colunas CO_ANO e CO_MES para inteiros. Posterior a isso a tabela exportações bem como as tabelas dimenções foram persistidas no banco de dados Prata.

In [0]:


expprata=spark.table("MVP.Bronze.Exportacoes")
expprata=expprata.withColumn("data", to_date(concat(col("CO_ANO"), col("CO_MES"), lit("01")), "yyyyMMdd"))
expprata=expprata.withColumn('QT_ESTAT', col('QT_ESTAT').cast("float"))
expprata=expprata.withColumn('KG_LIQUIDO', col('KG_LIQUIDO').cast("float"))
expprata=expprata.withColumn('VL_FOB', col('VL_FOB').cast("float"))
expprata=expprata.withColumn('CO_MES', col('CO_MES').cast("int"))
expprata=expprata.withColumn('CO_ANO', col('CO_ANO').cast("int"))
expprata.write.format("delta").mode("overwrite").saveAsTable("MVP.Prata.Exportacoes")

viaprata=spark.table("MVP.Bronze.DimVia")
viaprata.write.format("delta").mode("overwrite").saveAsTable("MVP.Prata.DimVia")

urfprata=spark.table("MVP.Bronze.DimURF")
urfprata.write.format("delta").mode("overwrite").saveAsTable("MVP.Prata.DimURF")

paisprata=spark.table("MVP.Bronze.DimPais")
paisprata.write.format("delta").mode("overwrite").saveAsTable("MVP.Prata.DimPais")

unidadeprata=spark.table("MVP.Bronze.DimUnidade")
unidadeprata.write.format("delta").mode("overwrite").saveAsTable("MVP.Prata.DimUnidade")

ncmprata=spark.table("MVP.Bronze.DimNCM")
ncmprata.write.format("delta").mode("overwrite").saveAsTable("MVP.Prata.DimNCM")





# Criação das Tabelas Ouro
Nesta etapa a tabela exportações foi filtrada para ter apenas os NCM que iniciam em 93, pois com isso possui apenas os NCM correspondentes a armamento que são o objetivo da analise, também foi criada uma tabela calendario. As demais tabelas não possuiam alterações a serem feitas.

In [0]:
spark.sql("DROP TABLE IF EXISTS MVP.Ouro.Calendario")
expouro=spark.table("MVP.Prata.Exportacoes")
expouro=expouro.filter(F.col("CO_NCM").startswith("93"))
expouro.write.format("delta").mode("overwrite").saveAsTable("MVP.Ouro.Exportacoes")

start='2000-01-01'
end='2026-12-31'
start_dt = datetime.strptime(start, "%Y-%m-%d")
end_dt   = datetime.strptime(end, "%Y-%m-%d")

num_dias = (end_dt - start_dt).days + 1

calendario = (
    spark.range(0, num_dias)
         .withColumn("date", F.expr(f"date_add('{start}', CAST(id AS INT))"))
         .withColumn("mes", F.date_format("date", "MMMM"))
)
calendario=calendario.withColumn("mes_num", F.month(calendario.date))
calendario=calendario.withColumn("ano", F.year(calendario.date))
calendario.write.format("delta").mode("overwrite").saveAsTable("MVP.Ouro.Calendario")

viaouro=spark.table("MVP.Prata.DimVia")
viaouro.write.format("delta").mode("overwrite").saveAsTable("MVP.Ouro.DimVia")

urfouro=spark.table("MVP.Prata.DimURF")
urfouro.write.format("delta").mode("overwrite").saveAsTable("MVP.Ouro.DimURF")

paisouro=spark.table("MVP.Prata.DimPais")
paisouro.write.format("delta").mode("overwrite").saveAsTable("MVP.Ouro.DimPais")

unidadeouro=spark.table("MVP.Prata.DimUnidade")[['CO_UNID', 'NO_UNID', 'SG_UNID']]
unidadeouro.write.format("delta").mode("overwrite").saveAsTable("MVP.Ouro.DimUnidade")

ncmouro=spark.table("MVP.Prata.DimNCM")
ncmouro.write.format("delta").mode("overwrite").saveAsTable("MVP.Ouro.DimNCM")

# Star Schema
Aqui foram realizados os joins da tabela fato com as dimenções seguindo o modelo do tipo star schema

In [0]:
expo=spark.table("MVP.Ouro.Exportacoes")
calendario=spark.table("MVP.Ouro.Calendario")
ncm=spark.table("MVP.Ouro.DimNCM")
via=spark.table("MVP.Ouro.DimVia")
urf=spark.table("MVP.Ouro.DimURF")
pais=spark.table("MVP.Ouro.DimPais")
unidade=spark.table("MVP.Ouro.DimUnidade")

e = expo.alias("e")
c = calendario.alias("c")
n = ncm.alias("n")
v = via.alias("v")
u = urf.alias("u")
p = pais.alias("p")
un = unidade.alias("un")

expo = (
    e.join(c, col("e.data") == col("c.date"), "left")
     .join(n, col("e.CO_NCM") == col("n.CO_NCM"), "left")
     .join(v, col("e.CO_VIA") == col("v.CO_VIA"), "left")
     .join(u, col("e.CO_URF") == col("u.CO_URF"), "left")
     .join(p, col("e.CO_PAIS") == col("p.CO_PAIS"), "left")
     .join(un, col("e.CO_UNID") == col("un.CO_UNID"), "left")
     .select(
         "e.*",
         "c.mes", "c.ano","c.mes_num",
         "n.NO_NCM_POR",
         "v.NO_VIA",
         "u.`NO_URF`",
         "p.NO_PAIS",
         "un.SG_UNID"
     )
)


expo.head()


Row(CO_ANO=2025, CO_MES=7, CO_NCM='93062100', CO_UNID='10', CO_PAIS='355', SG_UF_NCM='SP', CO_VIA='01', CO_URF='0817800', QT_ESTAT=14460.0, KG_LIQUIDO=14460.0, VL_FOB=229039.0, data=datetime.date(2025, 7, 1), mes='July', ano=2025, mes_num=7, NO_NCM_POR='Cartuchos para espingardas ou carabinas de cano liso', NO_VIA='MARITIMA', NO_URF='0817800 - PORTO DE SANTOS', NO_PAIS='Hungria', SG_UNID='KGL  ')

In [0]:
calendario.head()

Row(id=2465, date=datetime.date(2006, 10, 1), mes=10, ano=2006)

# Respondendo as Perguntas

## O valor global das exportaçoes de armamento tem mudado❔ Se sim, como❔Qual a tendencia atual❔
Sim o valor tem mudado, mostra uma tendencia crescente, apesas de ter caido nos ultimos dois anos os valores em 2023 e 2024 foram atipicamnte altos, assim sem a consolidação do ano de 2026 é cedo para dizer se há uma reverção na tendencia.

In [0]:
px.bar(expo.toPandas()[["ano","VL_FOB"]].groupby("ano").sum().reset_index(), x="ano", y="VL_FOB", labels={"VL_FOB": "Exportação em Dolares"}, title="Valor das Exportações por Ano")

### **Figura1-Valor das Exportações por Ano**

## Quais os três paises que historicamente mais compraram do Brasil❔ Percentualmente quanto isso representa de nossas vendas❔ Juntos mantem um padrão de compra percentual ao longo dos anos❔
Os três Paises que no geral mais compram do Brasil no periodo foram Estados Unidos, Emirados Arabes e Arabia Saúdita. Representando 61,9% do valor das compras, sendo os EUA o maior comprador de longe. Ao longo do tempo eles tendem a se manter comprando entre 50 a 70% das nossas vendas. Apesar de haver alguns anos claramente atipicos como 2002 e 2014 não houve tempo habil de estudar a razão disso no presente trabalho.

In [0]:
df=expo.toPandas()[["NO_PAIS","VL_FOB"]].groupby("NO_PAIS",as_index=False).sum().sort_values("VL_FOB",ascending=False).reset_index()
df['Percentual']=round(df['VL_FOB']/df['VL_FOB'].sum()*100,3)
df['País']=np.where(df.index>=3,"Outros",df['NO_PAIS'])
df=df[['País','Percentual']].groupby('País',as_index=False).sum().sort_values('Percentual',ascending=False).reset_index()
px.pie(df, names='País', values="Percentual",color='País',category_orders={'País':['Outros','Países com mais exportações']}, hole=.3, title="Percentual do Valor de Exportação por País")

### **Figura2-Percentual do Valor de Exportação por País**

In [0]:
anos=expo.toPandas()['ano'].unique()
dfs=[]
for ano in anos:
  df=expo.toPandas()
  df=df[df['ano']==ano]
  df=df[['NO_PAIS','VL_FOB','ano']]
  df=df.groupby(['NO_PAIS','ano'],as_index=False).sum().sort_values('VL_FOB',ascending=False).reset_index()
  df['Percentual']=round(df['VL_FOB']/df['VL_FOB'].sum()*100,3)
  df=df[df['NO_PAIS'].isin(['Estados Unidos','Arábia Saudita','Emirados Árabes Unidos'])]
  linha=pd.DataFrame({'ano':[ano],'NO_PAIS':'Total','VL_FOB':df['VL_FOB'].sum(),'Percentual':df['Percentual'].sum()})
  df=pd.concat([df,linha],ignore_index=True)
  dfs.append(df)

df=pd.concat(dfs,ignore_index=True)

px.line(df.sort_values('ano'), x='ano', y='Percentual', color='NO_PAIS',title="Percentual do Valor de Exportação por Ano",labels={"Percentual": "Percentual de Exportação"})



### **Figura3-Percentual do Valor de Exportação por Ano+Pais**

## Quais os três paises que historicamente mais compraram do Brasil❔ Percentualmente quanto isso representa de nossas vendas❔ Juntos mantem um padrão de compra percentual ao longo dos anos❔

Os tres produtos mais vendidos pelo Brasil foram "Outras espingardas e carabinas de caça ou de tiro ao alvo, com pelo menos um cano liso", "Outras espingardas e carabinas de caça ou de tiro ao alvo" e "Cartuchos para espingardas ou carabinas de cano liso", estes possuem como codigos NCM 93062100, 93033000 e 93032000 respectivamente. Quase todo o valor de venda do Brasil foi concentrado nos codigos 93062100 e 93033000 tanto no total do periodo analisado quanto no decorrer dele.

In [0]:
df=expo.toPandas()[["CO_NCM","VL_FOB"]].groupby("CO_NCM",as_index=False).sum().sort_values("VL_FOB",ascending=False).reset_index()
px.bar(df,x="CO_NCM",y='VL_FOB',color="CO_NCM",barmode='stack',title="Valor de Exportação por NCM",labels={"VL_FOB": "Exportação em Dolares"})

### **Figura4-Valor de Exportação por NCM**

In [0]:
anos=expo.toPandas()['ano'].unique()
dfs=[]
for ano in anos:
  df=expo.toPandas()
  df=df[df['ano']==ano]
  df=df[['CO_NCM','VL_FOB','ano']]
  df=df.groupby(['CO_NCM','ano'],as_index=False).sum().sort_values('VL_FOB',ascending=False).reset_index()
  df['Percentual']=round(df['VL_FOB']/df['VL_FOB'].sum()*100,3)
  df=df[df['CO_NCM'].isin(['93062100','93033000','93032000'])]
  linha=pd.DataFrame({'ano':[ano],'CO_NCM':'Total','VL_FOB':df['VL_FOB'].sum(),'Percentual':df['Percentual'].sum()})
  df=pd.concat([df,linha],ignore_index=True)
  dfs.append(df)

df=pd.concat(dfs,ignore_index=True)

px.line(df.sort_values('ano'), x='ano', y='Percentual', color='CO_NCM',title="Percentual do Valor de Exportação por Ano",labels={"Percentual": "Percentual de Exportação"})

### **Figura5-Percentual do Valor de Exportação por Ano+NCM**

In [0]:
%sql
SELECT CO_NCM, NO_NCM_POR FROM MVP.Ouro.DimNCM WHERE CO_NCM IN ('93062100','93033000','93032000')
    

    

CO_NCM,NO_NCM_POR
93032000,"Outras espingardas e carabinas de caça ou de tiro ao alvo, com pelo menos um cano liso"
93033000,Outras espingardas e carabinas de caça ou de tiro ao alvo
93062100,Cartuchos para espingardas ou carabinas de cano liso


## Historicamente quais as principais vias de exportação❔
As principais vias de exportação são a maritim que totalizaram 56,2% e Aerea com 40,9%. Ao longo dos anos elas sempre se mantem como as duas principais vias exportadoras.

In [0]:
df=expo.toPandas()[["VL_FOB",'ano','NO_VIA']].groupby(['ano','NO_VIA'],as_index=False).sum().sort_values("VL_FOB",ascending=False).reset_index()
px.pie(df,names='NO_VIA',values='VL_FOB',color='NO_VIA', title="Percentual do Valor de Exportação por Via de Exportação")



### **Figura6-Percentual do Valor de Exportação por Via de Exportação**

In [0]:
px.bar(df,x='ano',y='VL_FOB',color='NO_VIA',barmode='stack', labels={"VL_FOB": "Exportação em Dolares"}, title="Valor das Exportações por Ano")

### **Figura7-Valor das Exportações por Ano+Via**

## Existe algum mês do ano em que as vendas tendem a ocorrer mais❔
Não foi possivel identificar um mês ou meses especificos em qua as vendas estejam consistentemente mais concentradas

In [0]:
anos=expo.toPandas()['ano'].unique()
dfs=[]
for ano in anos:
  df=expo.toPandas()
  df=df[df['ano']==ano]
  df=df[['VL_FOB','ano','mes','mes_num']]
  df=df.groupby(['ano','mes','mes_num'],as_index=False).sum().reset_index()
  df['Percentual']=round(df['VL_FOB']/df['VL_FOB'].sum()*100,3)
  dfs.append(df)

df=pd.concat(dfs,ignore_index=True).sort_values(['ano','mes_num'],ascending=False)


px.bar(df,x="ano",y="Percentual",color="mes", labels={"Percentual": "Percentual de Exportação"},title="Percentual do Valor de Exportação por Ano/Mês")

## **Figura8-Percentual do Valor de Exportação por Ano+Mês**

# Auto Avaliação
Entende-se que foram propostas perguntas claras e objetivas com respostas satisfatoria na analise, a ingestão de dados foi feita de maneira eficiente e correta e a experiencia em nuvem realizada com sucesso. Assim considera-se o trabalho bem sucedido.